## Unilingual Model Exploration

This section explores unilingual models (Ensemble methods) that uses one model per language


---
Note that cross-validation process differs if we use a multi-lingual model or mono-lingual model:
- Multi-Lingual: Each fold should contain all the nodes with the same sentence_id and for all languages! (To avoid unbalance)
- Uni-Lingual: Each fold should contain all the the nodes with the same sentence_id. There are 2 ways to do this:

In [4]:
# EL CLASSICO: 
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt 

# Classical Scikit-Learn Imports
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier

# Advanced ML Imports
import optuna
from catboost import CatBoostClassifier, Pool
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# eZAutoML Imports
#from ezautoml.model import eZAutoML
#from ezautoml.space.search_space import SearchSpace
#from ezautoml.evaluation.metric import MetricSet, Metric
#from ezautoml.evaluation.task import TaskType
#from ezautoml.optimization.optimizers.random_search import RandomSearchOptimizer

# Custom libraries and functions
from src.unilingual_ensemble import UnilingualEnsembleClassifier
from src.cross_validation import run_groupkfold_cv
from src.submission import generate_kaggle_submission

def evaluate_model(y_true_df, y_pred_df):
    # Assuming y_true_df and y_pred_df are DataFrames with 'root' and matching length
    if 'root' not in y_true_df.columns or 'root' not in y_pred_df.columns:
        print("Error: 'root' column missing in one of the dataframes.")
        return
    if len(y_true_df) != len(y_pred_df):
        print("Error: Length mismatch between true and predicted values.")
        print(f"  Length of true values: {len(y_true_df)}")
        print(f"  Length of predicted values: {len(y_pred_df)}")
        return
        
    y_true = y_true_df['root']
    y_pred = y_pred_df['root']
    correct_predictions = (y_true == y_pred).sum()
    total_predictions = len(y_true)
    accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0
    print(f"Number of correct predictions: {correct_predictions} / {total_predictions}")
    print(f"Evaluation accuracy: {accuracy:.4f}")


# Load data
train = pd.read_csv("./data/train_dataset_ultraprocessed.csv")
test = pd.read_csv("./data/test_dataset_ultraprocessed.csv")
train

,sentence_id,language,node,sentence_length,root,degree,degree_squared,degree_diff,local_degree_ratio,max_neighbor_degree,...,local_entropy,global_entropy,eccentricity,closeness_centrality,kcore_number,avg_shortest_path_length,pca_1,pca_2,pca_3,cluster
0,2,Japanese,14,23,0,-0.178144,-0.214265,0.400779,-0.064361,-0.862076,...,0.197384,-0.267930,1.134275,-0.527223,-0.373383,0.849250,-0.726772,-0.639728,1.854627,0
1,2,Japanese,8,23,0,-0.178144,-0.214265,0.400779,-0.064361,-0.862076,...,0.197384,-0.267930,0.749748,-0.458162,-0.373383,0.310212,-0.609886,-0.935976,1.284601,0
2,2,Japanese,4,23,0,-0.803671,-0.673585,-0.078100,-0.559448,-0.862076,...,-0.888409,-0.267930,1.518802,-0.597152,-0.373383,1.578536,-2.356987,1.134445,2.251312,0
3,2,Japanese,6,23,0,-0.178144,-0.214265,0.400779,-0.064361,-0.351293,...,-0.007529,-0.267930,1.134275,-0.534229,-0.373383,0.912666,-0.746906,-0.410513,1.723765,0
4,2,Japanese,2,23,0,0.447383,0.551268,1.039284,0.727776,-0.862076,...,0.764089,-0.267930,0.749748,-0.458162,-0.373383,0.310212,0.414012,-1.678863,1.672658,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
197474,995,Russian,2,19,0,-0.671981,-0.641352,-0.758611,-0.654991,-0.028694,...,-0.888409,0.075185,-0.221689,-0.186386,-0.020751,0.207622,-1.644817,1.379274,-0.403877,1
197475,995,Russian,14,19,0,-0.671981,-0.641352,-1.918001,-0.814808,1.207936,...,-0.888409,0.075185,0.243791,-0.093511,-0.020751,-0.164092,-1.243046,2.636507,-1.697236,1
197476,995,Russian,5,19,0,0.085235,-0.085333,-0.178916,-0.255446,1.207936,...,-0.034032,0.075185,0.243791,-0.066927,-0.020751,-0.257020,0.020820,0.354585,-0.749208,1
197477,995,Russian,16,19,0,-0.671981,-0.641352,-0.178916,-0.455220,-0.647010,...,-0.888409,0.075185,0.709271,-0.253477,-0.020751,0.532871,-1.748921,1.085061,0.956803,1


### Model 1: Random Forest Ensemble

In [ ]:
target_col = "root"
group_col = "sentence_id" 


# Train final model on full train data
model = UnilingualEnsembleClassifier(
    base_model_cls=RandomForestClassifier,
    base_model_kwargs={'n_estimators': 100, 'n_jobs': 1},
    language_colname='language',
    gridsearch_per_language=True,
    cv=2,
    param_grid={
    'n_estimators': [250, 500],            # Number of trees
    'max_depth': [None, 10, 20],           # Tree depth
    'class_weight': ['balanced']         # For imbalanced classes
    },
    n_jobs=21
)
model.fit(train.drop(columns=target_col), train[target_col])

In [ ]:
# 1. Prepare metadata from test
test_meta = test[['sentence_id', 'node', 'language']].copy()

# 2. Remove target column from test if exists
X_test = test.drop(columns=[target_col]) if target_col in test.columns else test

# 3. Generate submission
generate_kaggle_submission(
    model=model,
    X_test=X_test,
    test_meta=test_meta,
    output_path="data/predictions_submission_rf_unilingual.csv",
    language_prefix="language_",  # Only relevant if X_test contains one-hot language columns
    y_true=test[target_col] if target_col in test.columns else None,
    return_df=False,
    verbose=True
)


## Model 2: XGBoost Ensemble

In [ ]:
target_col = "root"
group_col = "sentence_id" 

# Run GroupKFold CV on train
cv_scores = run_groupkfold_cv(
    X=train,
    y=train[target_col],
    group_colname=group_col,
    clf_cls=UnilingualEnsembleClassifier,
    clf_kwargs={
        'base_model_cls': XGBClassifier,
        'base_model_kwargs': {'n_estimators': 300, 'n_jobs': 1},
        'language_colname': 'language',
        'n_jobs': 8  # adjust based on your CPU
    },
    n_splits=5,
    metric_fn=accuracy_score,
    verbose=True
)

# Plot CV scores
plt.plot(cv_scores, marker='o')
plt.title('CV Accuracy Scores per Fold')
plt.xlabel('Fold')
plt.ylabel('Accuracy')
plt.grid(True)
plt.show()

# Train final model on full train data
model = UnilingualEnsembleClassifier(
    base_model_cls=XGBClassifier,
    base_model_kwargs={'n_estimators': 300, 'n_jobs': 1},
    language_colname='language',
    n_jobs=8
)
model.fit(train.drop(columns=target_col), train[target_col])

In [ ]:
# 1. Prepare metadata from test
test_meta = test[['sentence_id', 'node', 'language']].copy()
# 2. Remove target column from test if exists
X_test = test.drop(columns=[target_col]) if target_col in test.columns else test

# 3. Generate submission
generate_kaggle_submission(
    model=model,X_test=X_test,
    test_meta=test_meta,
    output_path="data/predictions_submission_unilingual_xgboost.csv",
    language_prefix="language_",  # Only relevant if X_test contains one-hot language columns
    y_true=test[target_col] if target_col in test.columns else None,
    return_df=False,
    verbose=True
)


## Model 3: LightGBM Ensemble

In [ ]:
target_col = "root"
group_col = "sentence_id" 


# Train final model on full train data
model = UnilingualEnsembleClassifier(
    base_model_cls=LGBMClassifier,
    base_model_kwargs={'n_estimators': 500, 'n_jobs': 1},
    language_colname='language',
    gridsearch_per_language=True,
    cv=2,
    param_grid={
    'max_depth': [None, 5, 10, 20],           # Tree depth; None allows full growth
    'class_weight': [None, 'balanced'],         # For imbalanced classes
    },
    n_jobs=21
)
model.fit(train.drop(columns=target_col), train[target_col])

[LightGBM] [Info] Number of positive: 250, number of negative: 3705
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000317 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4372
[LightGBM] [Info] Number of data points in the train set: 3955, number of used features: 25
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.063211 -> initscore=-2.695978
[LightGBM] [Info] Start training from score -2.695978
[LightGBM] [Info] Number of positive: 250, number of negative: 4063
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000389 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4497
[LightGBM] [Info] Number of data points in the train set: 4313, number of used features: 25
[LightGBM] [Info] [binary:

UnilingualEnsembleClassifier(base_model_cls=<class 'lightgbm.sklearn.LGBMClassifier'>,
                             base_model_kwargs={'n_estimators': 500,
                                                'n_jobs': 1},
                             gridsearch_per_language=True, n_jobs=21,
                             param_grid={'class_weight': [None, 'balanced'],
                                         'max_depth': [None, 5, 10, 20]})

In [ ]:
# 1. Prepare metadata from test
test_meta = test[['sentence_id', 'node', 'language']].copy()
# 2. Remove target column from test if exists
X_test = test.drop(columns=[target_col]) if target_col in test.columns else test

# 3. Generate submission
generate_kaggle_submission(
    model=model,X_test=X_test,
    test_meta=test_meta,
    output_path="data/predictions_submission_unilingual_lightgbm.csv",
    language_prefix="language_",  # Only relevant if X_test contains one-hot language columns
    y_true=test[target_col] if target_col in test.columns else None,
    return_df=False,
    verbose=True
)


2025-05-29 13:45:28.580 | INFO     | src.submission:generate_kaggle_submission:29 - Generating predictions...
2025-05-29 13:45:38.604 | INFO     | src.submission:generate_kaggle_submission:42 - Detected multilingual (per-language model ensemble) setup.
2025-05-29 13:45:38.649 | SUCCESS  | src.submission:generate_kaggle_submission:75 - Submission saved to: data/predictions_submission_unilingual_lightgbm.csv


In [ ]:
try:
    # Reload for this function to ensure clean state
    kaggle_perfect_predictions_eval = pd.read_csv("data/kaggle_perfect_predictions.csv")
    current_predictions_eval = pd.read_csv('data/predictions_submission_unilingual_lightgbm.csv')
    evaluate_model(kaggle_perfect_predictions_eval, current_predictions_eval)
except FileNotFoundError:
    print("One of the prediction files not found. Skipping custom evaluation.")
except Exception as e:
    print(f"An error occurred during custom evaluation: {e}")

Number of correct predictions: 2540 / 10395
Evaluation accuracy: 0.2443


## Model 4: SVM

In [ ]:
target_col = "root"
group_col = "sentence_id"


# Train final model on full train data
model = UnilingualEnsembleClassifier(
    base_model_cls=SVC,
    base_model_kwargs={"probability": True, "class_weight": "balanced"},
    language_colname='language',
    gridsearch_per_language=True,
    param_grid = {
        "C": [0.1, 1, 10],                     # Regularization strength
        "kernel": ["rbf"],          # Simpler and widely useful kernels
        "gamma": ["scale", "auto"],
    },
    cv=2,
    n_jobs=21
)
model.fit(train.drop(columns=target_col), train[target_col])

2025-05-31 08:42:37.094 | SUCCESS  | src.unilingual_ensemble:_fit_one:53 - Finished training model for language: Finnish
2025-05-31 08:43:15.423 | SUCCESS  | src.unilingual_ensemble:_fit_one:53 - Finished training model for language: Czech
2025-05-31 08:43:34.584 | SUCCESS  | src.unilingual_ensemble:_fit_one:53 - Finished training model for language: Turkish
2025-05-31 08:43:47.941 | SUCCESS  | src.unilingual_ensemble:_fit_one:53 - Finished training model for language: Swedish
2025-05-31 08:43:51.517 | SUCCESS  | src.unilingual_ensemble:_fit_one:53 - Finished training model for language: Icelandic
2025-05-31 08:44:06.104 | SUCCESS  | src.unilingual_ensemble:_fit_one:53 - Finished training model for language: Polish
2025-05-31 08:44:12.879 | SUCCESS  | src.unilingual_ensemble:_fit_one:53 - Finished training model for language: Russian
2025-05-31 08:44:20.143 | SUCCESS  | src.unilingual_ensemble:_fit_one:53 - Finished training model for language: Korean
2025-05-31 08:44:33.162 | SUCCESS 

UnilingualEnsembleClassifier(base_model_cls=<class 'sklearn.svm._classes.SVC'>,
                             base_model_kwargs={'class_weight': 'balanced',
                                                'probability': True},
                             gridsearch_per_language=True, n_jobs=21,
                             param_grid={'C': [0.1, 1, 10],
                                         'gamma': ['scale', 'auto'],
                                         'kernel': ['rbf']})

In [ ]:
# 1. Prepare metadata from test
test_meta = test[['sentence_id', 'node', 'language']].copy()
# 2. Remove target column from test if exists
X_test = test.drop(columns=[target_col]) if target_col in test.columns else test

# 3. Generate submission
generate_kaggle_submission(
    model=model,X_test=X_test,
    test_meta=test_meta,
    output_path="data/predictions_submission_unilingual_svm_tuned.csv",
    language_prefix="language_",  # Only relevant if X_test contains one-hot language columns
    y_true=test[target_col] if target_col in test.columns else None,
    return_df=False,
    verbose=True
)


2025-05-31 08:46:38.327 | INFO     | src.submission:generate_kaggle_submission:42 - Generating predictions...


2025-05-31 08:46:50.666 | INFO     | src.submission:generate_kaggle_submission:55 - Detected multilingual (per-language model ensemble) setup.
2025-05-31 08:46:50.693 | SUCCESS  | src.submission:generate_kaggle_submission:88 - Submission saved to: data/predictions_submission_unilingual_svm_tuned.csv


In [1]:
def evaluate_model(y_true_df, y_pred_df):
    # Assuming y_true_df and y_pred_df are DataFrames with 'root' and matching length
    if 'root' not in y_true_df.columns or 'root' not in y_pred_df.columns:
        print("Error: 'root' column missing in one of the dataframes.")
        return
    if len(y_true_df) != len(y_pred_df):
        print("Error: Length mismatch between true and predicted values.")
        print(f"  Length of true values: {len(y_true_df)}")
        print(f"  Length of predicted values: {len(y_pred_df)}")
        return
        
    y_true = y_true_df['root']
    y_pred = y_pred_df['root']
    correct_predictions = (y_true == y_pred).sum()
    total_predictions = len(y_true)
    accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0
    print(f"Number of correct predictions: {correct_predictions} / {total_predictions}")
    print(f"Evaluation accuracy: {accuracy:.4f}")

try:
    # Reload for this function to ensure clean state
    kaggle_perfect_predictions_eval = pd.read_csv("data/kaggle_perfect_predictions.csv")
    current_predictions_eval = pd.read_csv('data/predictions_submission_unilingual_svm_tuned.csv')
    evaluate_model(kaggle_perfect_predictions_eval, current_predictions_eval)
except FileNotFoundError:
    print("One of the prediction files not found. Skipping custom evaluation.")
except Exception as e:
    print(f"An error occurred during custom evaluation: {e}")

An error occurred during custom evaluation: name 'pd' is not defined


### Model 5: eZAutoML Exploration

In [ ]:
train = pd.read_csv("./data/train_dataset_ultraprocessed.csv")
test = pd.read_csv("./data/test_dataset_ultraprocessed.csv")

# === Set target and group columns ===
target_col = "root"
group_col = "sentence_id"  # not needed for this run, unless using grouped CV

# === Extract features and target ===
X = train.drop(columns=[target_col])
y = train[target_col]

# === Train/test split (optional if using full train set) ===
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# === Define metrics ===
metrics = MetricSet(
    {"accuracy": Metric(name="accuracy", fn=accuracy_score, minimize=False)},
    primary_metric_name="accuracy"
)

# === Define search space ===
search_space = SearchSpace.from_builtin("classification_space")

# === Initialize eZAutoML ===
ezautoml = eZAutoML(
    search_space=search_space,
    task=TaskType.CLASSIFICATION,
    metrics=metrics,
    max_trials=50,     # You can increase for better models
    max_time=600,      # 10 minutes
    seed=42
)

# === Fit the AutoML model ===
ezautoml.fit(X_train, y_train)

# === Evaluate ===
test_accuracy = ezautoml.test(X_test, y_test)
print("Test accuracy:", test_accuracy)

# === Summary of best models ===
ezautoml.summary(k=50)


ModuleNotFoundError: No module named 'torch'

## Model 6: HistogramBoosting (UNI)

In [2]:
# Load data
train = pd.read_csv("./data/train_dataset_ultraprocessed.csv")
test = pd.read_csv("./data/test_dataset_ultraprocessed.csv")

# Make sure 'root' is your target and 'sentence_id' groups sentences (adjust if needed)
target_col = "root"
group_col = "sentence_id"  # replace if different in your dataset


# Train final model on full train data
model = UnilingualEnsembleClassifier(
    base_model_cls=HistGradientBoostingClassifier,
    base_model_kwargs={"early_stopping": True},
    language_colname='language',
    gridsearch_per_language=True,
    cv=2,
    param_grid = {
        'max_depth': [None, 6, 10],
        'learning_rate': [0.01, 0.05, 0.1],
        'max_iter': [100, 300, 500],
        'l2_regularization': [0.01, 0.1, 1],
        'min_samples_leaf': [20, 50, 100],
    },
    n_jobs=21
)
model.fit(train.drop(columns=target_col), train[target_col])

UnilingualEnsembleClassifier(base_model_cls=<class 'sklearn.ensemble._hist_gradient_boosting.gradient_boosting.HistGradientBoostingClassifier'>,
                             base_model_kwargs={'early_stopping': True},
                             gridsearch_per_language=True, n_jobs=21,
                             param_grid={'l2_regularization': [0.01, 0.1, 1],
                                         'learning_rate': [0.01, 0.05, 0.1],
                                         'max_depth': [None, 6, 10],
                                         'max_iter': [100, 300, 500],
                                         'min_samples_leaf': [20, 50, 100]})

In [ ]:
# 1. Prepare metadata from test
test_meta = test[['sentence_id', 'node', 'language']].copy()
# 2. Remove target column from test if exists
X_test = test.drop(columns=[target_col]) if target_col in test.columns else test

# 3. Generate submission
generate_kaggle_submission(
    model=model,X_test=X_test,
    test_meta=test_meta,
    output_path="data/predictions_submission_unilingual_hist.csv",
    language_prefix="language_",  # Only relevant if X_test contains one-hot language columns
    y_true=test[target_col] if target_col in test.columns else None,
    return_df=False,
    verbose=True
)


2025-05-29 20:05:31.560 | INFO     | src.submission:generate_kaggle_submission:42 - Generating predictions...
2025-05-29 20:05:34.423 | INFO     | src.submission:generate_kaggle_submission:55 - Detected multilingual (per-language model ensemble) setup.
2025-05-29 20:05:34.437 | SUCCESS  | src.submission:generate_kaggle_submission:88 - Submission saved to: data/predictions_submission_unilingual_hist.csv


In [10]:
def evaluate_model(y_true_df, y_pred_df):
    # Assuming y_true_df and y_pred_df are DataFrames with 'root' and matching length
    if 'root' not in y_true_df.columns or 'root' not in y_pred_df.columns:
        print("Error: 'root' column missing in one of the dataframes.")
        return
    if len(y_true_df) != len(y_pred_df):
        print("Error: Length mismatch between true and predicted values.")
        print(f"  Length of true values: {len(y_true_df)}")
        print(f"  Length of predicted values: {len(y_pred_df)}")
        return
        
    y_true = y_true_df['root']
    y_pred = y_pred_df['root']
    correct_predictions = (y_true == y_pred).sum()
    total_predictions = len(y_true)
    accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0
    print(f"Number of correct predictions: {correct_predictions} / {total_predictions}")
    print(f"Evaluation accuracy: {accuracy:.4f}")

try:
    # Reload for this function to ensure clean state
    kaggle_perfect_predictions_eval = pd.read_csv("data/kaggle_perfect_predictions.csv")
    current_predictions_eval = pd.read_csv('data/predictions_submission_unilingual_hist.csv')
    evaluate_model(kaggle_perfect_predictions_eval, current_predictions_eval)
except FileNotFoundError:
    print("One of the prediction files not found. Skipping custom evaluation.")
except Exception as e:
    print(f"An error occurred during custom evaluation: {e}")

Number of correct predictions: 3055 / 10395
Evaluation accuracy: 0.2939


### 7. AdaBoosting (UNI)

In [ ]:
# Load data
train = pd.read_csv("./data/train_dataset_ultraprocessed.csv")
test = pd.read_csv("./data/test_dataset_ultraprocessed.csv")

# Make sure 'root' is your target and 'sentence_id' groups sentences (adjust if needed)
target_col = "root"
group_col = "sentence_id"  # replace if different in your dataset


# Train final model on full train data
model = UnilingualEnsembleClassifier(
    base_model_cls=AdaBoostClassifier,
    base_model_kwargs={"estimator": DecisionTreeClassifier(), "n_estimators": 300},
    language_colname='language',
    gridsearch_per_language=True,
    cv=2,
    param_grid={
        'learning_rate': [0.01, 0.1, 1.0],
        'estimator__max_depth': [None, 10, 20],
    },
    n_jobs=21
)

model.fit(train.drop(columns=target_col), train[target_col])

In [ ]:
# 1. Prepare metadata from test
test_meta = test[['sentence_id', 'node', 'language']].copy()
# 2. Remove target column from test if exists
X_test = test.drop(columns=[target_col]) if target_col in test.columns else test

# 3. Generate submission
generate_kaggle_submission(
    model=model,X_test=X_test,
    test_meta=test_meta,
    output_path="data/predictions_submission_unilingual_adabooster.csv",
    language_prefix="language_",  # Only relevant if X_test contains one-hot language columns
    y_true=test[target_col] if target_col in test.columns else None,
    return_df=False,
    verbose=True
)

def evaluate_model(y_true_df, y_pred_df):
    # Assuming y_true_df and y_pred_df are DataFrames with 'root' and matching length
    if 'root' not in y_true_df.columns or 'root' not in y_pred_df.columns:
        print("Error: 'root' column missing in one of the dataframes.")
        return
    if len(y_true_df) != len(y_pred_df):
        print("Error: Length mismatch between true and predicted values.")
        print(f"  Length of true values: {len(y_true_df)}")
        print(f"  Length of predicted values: {len(y_pred_df)}")
        return
        
    y_true = y_true_df['root']
    y_pred = y_pred_df['root']
    correct_predictions = (y_true == y_pred).sum()
    total_predictions = len(y_true)
    accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0
    print(f"Number of correct predictions: {correct_predictions} / {total_predictions}")
    print(f"Evaluation accuracy: {accuracy:.4f}")

try:
    # Reload for this function to ensure clean state
    kaggle_perfect_predictions_eval = pd.read_csv("data/kaggle_perfect_predictions.csv")
    current_predictions_eval = pd.read_csv('data/predictions_submission_unilingual_adabooster.csv')
    evaluate_model(kaggle_perfect_predictions_eval, current_predictions_eval)
except FileNotFoundError:
    print("One of the prediction files not found. Skipping custom evaluation.")
except Exception as e:
    print(f"An error occurred during custom evaluation: {e}")

### 8. Catboost (UNI)

In [ ]:
# Load data
train = pd.read_csv("./data/train_dataset_ultraprocessed.csv")
test = pd.read_csv("./data/test_dataset_ultraprocessed.csv")

# Make sure 'root' is your target and 'sentence_id' groups sentences (adjust if needed)
target_col = "root"
group_col = "sentence_id"  # replace if different in your dataset

# Train final model on full train data
model = UnilingualEnsembleClassifier(
    base_model_cls=CatBoostClassifier,
    base_model_kwargs={"auto_class_weights": "Balanced"},
    language_colname='language',
    gridsearch_per_language=True,
    cv=2,
    param_grid = {
        'depth': [4, 6],
        'learning_rate': [0.03, 0.1]
    },
    n_jobs=21
)

model.fit(train.drop(columns=target_col), train[target_col])

In [ ]:
# 1. Prepare metadata from test
test_meta = test[['sentence_id', 'node', 'language']].copy()
# 2. Remove target column from test if exists
X_test = test.drop(columns=[target_col]) if target_col in test.columns else test

# 3. Generate submission
generate_kaggle_submission(
    model=model,X_test=X_test,
    test_meta=test_meta,
    output_path="data/predictions_submission_unilingual_catboost.csv",
    language_prefix="language_",  # Only relevant if X_test contains one-hot language columns
    y_true=test[target_col] if target_col in test.columns else None,
    return_df=False,
    verbose=True
)

def evaluate_model(y_true_df, y_pred_df):
    # Assuming y_true_df and y_pred_df are DataFrames with 'root' and matching length
    if 'root' not in y_true_df.columns or 'root' not in y_pred_df.columns:
        print("Error: 'root' column missing in one of the dataframes.")
        return
    if len(y_true_df) != len(y_pred_df):
        print("Error: Length mismatch between true and predicted values.")
        print(f"  Length of true values: {len(y_true_df)}")
        print(f"  Length of predicted values: {len(y_pred_df)}")
        return
        
    y_true = y_true_df['root']
    y_pred = y_pred_df['root']
    correct_predictions = (y_true == y_pred).sum()
    total_predictions = len(y_true)
    accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0
    print(f"Number of correct predictions: {correct_predictions} / {total_predictions}")
    print(f"Evaluation accuracy: {accuracy:.4f}")

try:
    # Reload for this function to ensure clean state
    kaggle_perfect_predictions_eval = pd.read_csv("data/kaggle_perfect_predictions.csv")
    current_predictions_eval = pd.read_csv('data/predictions_submission_unilingual_catboost.csv')
    evaluate_model(kaggle_perfect_predictions_eval, current_predictions_eval)
except FileNotFoundError:
    print("One of the prediction files not found. Skipping custom evaluation.")
except Exception as e:
    print(f"An error occurred during custom evaluation: {e}")

### 9. Catboost (MULTI)

#### Optuna Bayesian TPE Optimization

In [ ]:
 def objective(trial):
    params = {
        'depth': trial.suggest_int('depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
        'iterations': 300,
        'random_seed': 42,
        'auto_class_weights': 'Balanced'
    }
    
    X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2)
    
    model = CatBoostClassifier(**params, verbose=0)
    model.fit(X_train, y_train, eval_set=(X_valid, y_valid), early_stopping_rounds=30)
    
    preds = model.predict(X_valid)
    return accuracy_score(y_valid, preds)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)
print(study.best_params)


In [ ]:
# Load data
train = pd.read_csv("./data/train_dataset_ultraprocessed.csv")
test = pd.read_csv("./data/test_dataset_ultraprocessed.csv")

target_col = "root"
categorical_cols = ["language"]  # Add more if you have them

#best_params = {iterations=300, depth=6, learning_rate=0.1, auto_class_weights="Balanced"}
best_params = study.best_params
# Define Pool for training
train_pool = Pool(
    data=train.drop(columns=target_col),
    label=train[target_col],
    cat_features=categorical_cols
)

# Define the CatBoost model
model = CatBoostClassifier(**best_params)
# Train the model
model.fit(train_pool)


In [ ]:
# 1. Prepare metadata from test
test_meta = test[['sentence_id', 'node', 'language']].copy()
# 2. Remove target column from test if exists
X_test = test.drop(columns=[target_col]) if target_col in test.columns else test

# 3. Generate submission
generate_kaggle_submission(
    model=model,X_test=X_test,
    test_meta=test_meta,
    output_path="data/predictions_submission_multilingual_catboost.csv",
    language_prefix="language_",  # Only relevant if X_test contains one-hot language columns
    y_true=test[target_col] if target_col in test.columns else None,
    return_df=False,
    verbose=True
)

def evaluate_model(y_true_df, y_pred_df):
    # Assuming y_true_df and y_pred_df are DataFrames with 'root' and matching length
    if 'root' not in y_true_df.columns or 'root' not in y_pred_df.columns:
        print("Error: 'root' column missing in one of the dataframes.")
        return
    if len(y_true_df) != len(y_pred_df):
        print("Error: Length mismatch between true and predicted values.")
        print(f"  Length of true values: {len(y_true_df)}")
        print(f"  Length of predicted values: {len(y_pred_df)}")
        return
        
    y_true = y_true_df['root']
    y_pred = y_pred_df['root']
    correct_predictions = (y_true == y_pred).sum()
    total_predictions = len(y_true)
    accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0
    print(f"Number of correct predictions: {correct_predictions} / {total_predictions}")
    print(f"Evaluation accuracy: {accuracy:.4f}")

try:
    # Reload for this function to ensure clean state
    kaggle_perfect_predictions_eval = pd.read_csv("data/kaggle_perfect_predictions.csv")
    current_predictions_eval = pd.read_csv('data/predictions_submission_multilingual_catboost.csv')
    evaluate_model(kaggle_perfect_predictions_eval, current_predictions_eval)
except FileNotFoundError:
    print("One of the prediction files not found. Skipping custom evaluation.")
except Exception as e:
    print(f"An error occurred during custom evaluation: {e}")

### 10 Multi-Layer Perceptron

In [ ]:
model = UnilingualEnsembleClassifier(
    base_model_cls=MLPClassifier,
    base_model_kwargs = {
        'hidden_layer_sizes': (512, 256, 128, 64),  # a fairly large MLP
        'activation': 'relu',
        'alpha': 0.1, # L2 regularization strenght (This is ok) 
        'max_iter': 100,
        'early_stopping': True,
        'solver': 'adam',
        'batch_size': 'auto',
        'random_state': 42,
    },
    language_colname='language',
    gridsearch_per_language=True,  # Enable grid search only on learning rate
    cv=3,
    param_grid={
        'learning_rate_init': [0.0001, 0.001, 0.01]
    },
    n_jobs=8
)

model.fit(train.drop(columns=target_col), train[target_col])


In [ ]:
# 1. Prepare metadata from test
test_meta = test[['sentence_id', 'node', 'language']].copy()
# 2. Remove target column from test if exists
X_test = test.drop(columns=[target_col]) if target_col in test.columns else test

# 3. Generate submission
generate_kaggle_submission(
    model=model,X_test=X_test,
    test_meta=test_meta,
    output_path="data/predictions_submission_unilingual_mlp.csv",
    language_prefix="language_",  # Only relevant if X_test contains one-hot language columns
    y_true=test[target_col] if target_col in test.columns else None,
    return_df=False,
    verbose=True
)

def evaluate_model(y_true_df, y_pred_df):
    # Assuming y_true_df and y_pred_df are DataFrames with 'root' and matching length
    if 'root' not in y_true_df.columns or 'root' not in y_pred_df.columns:
        print("Error: 'root' column missing in one of the dataframes.")
        return
    if len(y_true_df) != len(y_pred_df):
        print("Error: Length mismatch between true and predicted values.")
        print(f"  Length of true values: {len(y_true_df)}")
        print(f"  Length of predicted values: {len(y_pred_df)}")
        return
        
    y_true = y_true_df['root']
    y_pred = y_pred_df['root']
    correct_predictions = (y_true == y_pred).sum()
    total_predictions = len(y_true)
    accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0
    print(f"Number of correct predictions: {correct_predictions} / {total_predictions}")
    print(f"Evaluation accuracy: {accuracy:.4f}")

try:
    # Reload for this function to ensure clean state
    kaggle_perfect_predictions_eval = pd.read_csv("data/kaggle_perfect_predictions.csv")
    current_predictions_eval = pd.read_csv('data/predictions_submission_unilingual_mlp.csv')
    evaluate_model(kaggle_perfect_predictions_eval, current_predictions_eval)
except FileNotFoundError:
    print("One of the prediction files not found. Skipping custom evaluation.")
except Exception as e:
    print(f"An error occurred during custom evaluation: {e}")